# Nemotron LoRA — train locally on your own GPU

**Works best on Linux (or WSL2) with an NVIDIA GPU that is sm_80+ and >=24 GB** (RTX 3090/4090, A-series). A true GTX (sm_61/75, <=11 GB) CANNOT run this model.

**Prereqs:** a Python 3.10-3.12 env with a CUDA build of **torch 2.7+** already installed for your CUDA. Windows note: `mamba_ssm` has no Windows wheels — use WSL2.

Run cell 0 first: it prints a clear GO / NO-GO verdict for your GPU.

## 0. Hardware check + verdict

In [ ]:
import torch
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {name}  sm_{cap[0]}{cap[1]}  {vram:.1f} GiB')
print('torch', torch.__version__, 'abi', torch.compiled_with_cxx11_abi())
ok_kernels = cap >= (8, 0)
ok_vram = vram >= 22
if ok_kernels and ok_vram:
    print('VERDICT: GO — fast Mamba kernels + enough VRAM for 4-bit.')
elif cap >= (8,0) and vram >= 11:
    print('VERDICT: TIGHT — kernels OK but VRAM<22GB; needs CPU offload (slow). Try it.')
else:
    print('VERDICT: NO-GO — sm<80 (kernels wont compile -> ~hours/step) and/or VRAM too small.')
    print('  This model needs an sm_80+ (Ampere/Ada) GPU with ~24GB. A GTX cannot train it.')

## 1. Get the code + dependencies
Assumes torch is already installed for your CUDA. We add the training stack only.

In [ ]:
!git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
%cd repo
!pip install -q "transformers>=4.45,<5" peft trl datasets accelerate bitsandbytes psutil einops hf_transfer

## 2. mamba_ssm + causal_conv1d matching your torch (Linux)
Fetches prebuilt wheels for your exact torch/CUDA/abi (needs torch 2.7+, abi=TRUE).

In [ ]:
!python kaggle_wheels/fetch_torch_locked_wheels.py --dest /tmp/mw
!pip install -q --no-deps /tmp/mw/causal_conv1d-*.whl /tmp/mw/mamba_ssm-*.whl
!python -c "import causal_conv1d, mamba_ssm; print('mamba OK')"

## 3. Hugging Face login + competition data
Accept the model license on its HF page if gated. Paste your Kaggle token for the data.

In [ ]:
import os, glob, shutil, urllib.request
os.makedirs('data', exist_ok=True)
hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
if hits:
    shutil.copy(hits[0], 'data/train.csv'); print('train.csv (mount) <-', hits[0])
else:
    TOK = "KGAT_xxxxxxxxxxxxxxxx"   # <-- your Kaggle API token (Colab/local path)
    url='https://www.kaggle.com/api/v1/competitions/data/download/nvidia-nemotron-model-reasoning-challenge/train.csv'
    req=urllib.request.Request(url, headers={'Authorization': f'Bearer {TOK}'})
    open('data/train.csv','wb').write(urllib.request.urlopen(req).read())
    print('train.csv (download):', os.path.getsize('data/train.csv'), 'bytes')

## 4. Build the SFT data (CPU)

In [ ]:
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data

## 5. Train (4-bit QLoRA; auto-offload + torch_forward if needed)
Set MEM_GIB to ~ (your VRAM - 2). On <22 GB it offloads to CPU RAM (slow). The trainer auto-forces torch_forward on sm<80 and patches the MoE dtype.

In [ ]:
import os, torch
vram = torch.cuda.get_device_properties(0).total_memory/1024**3
os.environ['QUANT'] = '4bit'
os.environ['NEMOTRON_MAX_MEMORY_GPU'] = f'{int(max(6, vram-2))}GiB'
os.environ['SFT_MAX_SEQ_LENGTH'] = '1024'
os.environ['NUM_EPOCHS'] = '1'
!python scripts/03_train_lora.py --data-path data/train_sft.jsonl --output-dir lora_adapter

## 6. Package locally -> submission.zip (no Kaggle round-trip)

In [ ]:
!python scripts/05_package_submission.py --adapter-dir lora_adapter --output submission.zip
print('Upload submission.zip to the competition.')